In [2]:
import os 
import time 
import json 
import uuid

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

load_dotenv(dotenv_path="../.env")

PORTKEY_API_KEY= os.getenv("PORTKEY_API_KEY")
portkey = Portkey(api_key=PORTKEY_API_KEY)

## Vırtual Key and Slugs

In [4]:
GROQ_SLUG = "gkey"
GROQ_MODEL = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

GROQ_SLUG_2 = "gkey2"
GROQ_OTHER_MODEL = f"@{"GROQ_SLUG_2"}/llama-3.1-8b-instant"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
from colorama import init, Fore, Style

init(autoreset=True)

print(f"\n{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}{Style.BRIGHT}🚀 SETUP COMPLETED SUCCESSFULLY")
print(f"{Fore.GREEN}{'='*60}\n")

status = f"{Fore.GREEN}✔ OK" if PORTKEY_API_KEY else f"{Fore.RED}✘ MISSING"

print(f"{Fore.CYAN}{'Portkey API Key':<22}{Fore.WHITE}: {status}")
print(f"{Fore.CYAN}{'Groq Slug':<22}{Fore.WHITE}: {Fore.YELLOW}{GROQ_SLUG}")
print(f"{Fore.CYAN}{'Groq Model':<22}{Fore.WHITE}: {Fore.YELLOW}{GROQ_MODEL}")
print(f"{Fore.CYAN}{'Groq Slug 2':<22}{Fore.WHITE}: {Fore.YELLOW}{GROQ_SLUG_2}")
print(f"{Fore.CYAN}{'Small Model':<22}{Fore.WHITE}: {Fore.YELLOW}{GROQ_OTHER_MODEL}")
print(f"{Fore.CYAN}{'Gateway URL':<22}{Fore.WHITE}: {Fore.BLUE}{PORTKEY_GATEWAY_URL}")

print(f"\n{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}✅ Environment is ready.")
print(f"{Fore.GREEN}{'='*60}")


🚀 SETUP COMPLETED SUCCESSFULLY

Portkey API Key       : ✔ OK
Groq Slug             : gkey
Groq Model            : @gkey/llama-3.3-70b-versatile
Groq Slug 2           : gkey2
Small Model           : @GROQ_SLUG_2/llama-3.1-8b-instant
Gateway URL           : https://api.portkey.ai/v1

✅ Environment is ready.


## Helper

In [6]:
def section(title):
    print(f"\n{"=" *55}")
    print(f"  {title}")
    print(f"{"=" *55}")

section("LLM Gateways")
    


  LLM Gateways


In [17]:
def show(q, answer, ms, Label=""):
    bar = chr(9472) * 55

    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")

    note = f" | {Label}" if Label else ""
    print(f"⏱ {ms:.0f}ms{note}")

    print(bar)


show(
    "What are the benefits of using Retrieval-Augmented Generation (RAG)?",
    "Retrieval-Augmented Generation (RAG) combines large language models with external knowledge sources, allowing responses to be more accurate, up-to-date, and grounded in retrieved documents instead of relying only on the model's internal knowledge.",
    12,
    "RAG Demo"
)


───────────────────────────────────────────────────────
Q: What are the benefits of using Retrieval-Augmented Generation (RAG)?
A: Retrieval-Augmented Generation (RAG) combines large language models with external knowledge sources, allowing responses to be more accurate, up-to-date, and grounded in retrieved documents instead of relying only on the model's internal knowledge.
⏱ 12ms | RAG Demo
───────────────────────────────────────────────────────


In [18]:
from langchain_groq import ChatGroq 
from langchain_core.messages import HumanMessage 

my_groq = ChatGroq(api_key=GROQ_API_KEY, model = "llama-3.3-70b-versatile", temperature=0)

In [19]:
section("BASELINE - GROQ Call")

questions = [
        "Write a funny joke about cats.",
        "Why does Python exist in this world?"
]

for q in questions:
    t0 = time.time()
    r = my_groq.invoke([HumanMessage(content = q)])
    show(q, r.content, (time.time()-t0)*1000, Label= "Groq Cal - without Gateway")


  BASELINE - GROQ Call

───────────────────────────────────────────────────────
Q: Write a funny joke about cats.
A: Why did the cat join a band? Because it wanted to be the purr-cussionist!
⏱ 673ms | Groq Cal - without Gateway
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
Q: Why does Python exist in this world?
A: Python exists in this world because of the vision and efforts of its creator, Guido van Rossum. In the late 1980s, van Rossum, a Dutch computer programmer, was working at the National Research Institute for Mathematics and Computer Science in the Netherlands. ...
⏱ 1972ms | Groq Cal - without Gateway
───────────────────────────────────────────────────────


## Portkey

In [21]:
section("Basic Portkey Call")

questions = [
    "What is AI and LLM?",
    "What is coffee and orange?"
]

for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model = GROQ_MODEL,
        messages=[{"role": "user", "content":q}]
    )

    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
        Label="Routed via Portkey Gateways")

print("\n Check Portkey AI >> Logs to see both requests fully logged!")
print("    Token Count, cost, latency- all tracked. Zero extra code.")


  Basic Portkey Call

───────────────────────────────────────────────────────
Q: What is AI and LLM?
A: **AI: Artificial Intelligence**
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. Learning
2. Problem-solving
3. Reasoning
4. Perception
5. Understanding la...
⏱ 2387ms | Routed via Portkey Gateways
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
Q: What is coffee and orange?
A: Coffee and orange is an interesting combination. While it may seem unusual, some people enjoy pairing coffee with orange flavors or ingredients. Here are a few possible interpretations:

1. **Coffee with orange flavor**: Some coffee drinks or syrups may have a...
⏱ 966ms | Routed via Portkey Gateways
───────────────────────────────────────────────────────

 Check Portkey AI >> Logs to see both requests fully logged!
    Token Count, cost, lat

In [22]:
section("Metadata & Observability")

# new unique id

session = str(uuid.uuid4())[:8]

scenarios = [
    ("mustafa", "enterprise-rag",  "Explain how RAG improves LLM accuracy."),
    ("john",    "security-agent",  "What is prompt injection and how can it be prevented?"),
    ("sarah",   "code-assistant",  "Generate a FastAPI endpoint for user authentication."),
    ("mustafa", "enterprise-rag",  "Compare semantic search with hybrid search."),
]



  Metadata & Observability


In [23]:
for user, feature, q in scenarios:
    t0 = time.time()
    r = portkey.with_options(
        metadata={
            "_user":       user,          # powers per-user analytics in dashboard
            "session_id":  session,
            "feature":     feature,
            "environment": "notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")


👤 mustafa  | 🔧 enterprise-rag     | 2377ms
  Q: Explain how RAG improves LLM accuracy.
  A: RAG (Retrieval-Augmented Generation) is a technique that improves the accuracy of Large Language Models (LLMs) by combin...

👤 john     | 🔧 security-agent     | 1533ms
  Q: What is prompt injection and how can it be prevented?
  A: Prompt injection is a type of attack that involves manipulating the input or prompts given to a language model or other ...

👤 sarah    | 🔧 code-assistant     | 2473ms
  Q: Generate a FastAPI endpoint for user authentication.
  A: **FastAPI User Authentication Endpoint**

Below is an example of a FastAPI endpoin...

👤 mustafa  | 🔧 enterprise-rag     | 2305ms
  Q: Compare semantic search with hybrid search.
  A: Semantic search and hybrid search are two advanced search technologies that aim to improve the way we search and retriev...


## Retries

In [31]:
retry_config = {
    "retry": {
        "attempts": 3,
        "on_status_codes": [429, 500, 502, 503, 504]
    }
}

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, config="pc-ai-sec-bd2117")

section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a AI?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")


  EXP 3 — Automatic Retries
Config: 3 retry attempts on [429, 500, 502, 503, 504]
Retries fire automatically on failure — transparent to your code

✅ Succeeded in 1129ms
   **Artificial Intelligence (AI) Definition:**

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

* Learning
* Problem-solving
* Reasoning
* Perception
* Understanding language

**Key Characteristics o

Retry sequence if Groq had failed:
  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3
  Your code only sees the final success or the last failure


## Timeouts

In [30]:
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms

portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config="pc-ai-sec-bd2117")

section(" Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")

print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))


   Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.

✅ Response in 1182ms (within 10s timeout)
   Kubernetes networking allows pods to communicate with each other and with the outside world through a complex system of virtual networks, network policies, and services, which enable the creation of a secure and scalable network infrastructure for containerized applications. The Kubernetes networking model is based on a flat network architecture, where each pod is assigned an IP address and can communicate with other pods and services without the need for explicit routing or NAT configuration.

--- Combining timeout + retry (production pattern) ---
{
  "request_timeout": 10000,
  "retry": {
    "attempts": 2,
    "on_status_codes": [
      408,
      429,
      503
    ]
  }
}


## Fallbacks

In [35]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},        # primary
        {"override_params": {"model": GROQ_OTHER_MODEL}}   # fallback if primary fails
    ]
}


portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config="pc-ai-sec-bd2117")


section("EXP 5 — Fallback Routing")
print(f"Primary  : {GROQ_MODEL}")
print(f"Fallback : {GROQ_OTHER_MODEL}\n")

fallback_questions = [
    "What is Intel Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, Label="primary served")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")




  EXP 5 — Fallback Routing
Primary  : @gkey/llama-3.3-70b-versatile
Fallback : @GROQ_SLUG_2/llama-3.1-8b-instant


───────────────────────────────────────────────────────
Q: What is Intel Technology?
A: Intel Technology refers to the innovative products, solutions, and innovations developed by Intel Corporation, a multinational corporation and technology company. Intel is one of the world's largest and most successful semiconductor chip makers, and its techno...
⏱ 3941ms | primary served
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
Q: Explain Kubernetes persistent volume claims.
A: Kubernetes Persistent Volume Claims (PVCs) are a crucial component in managing persistent storage for applications running in a Kubernetes cluster.

**What is a Persistent Volume Claim (PVC)?**

A Persistent Volume Claim (PVC) is a request for storage resource...
⏱ 2338ms | primary served
────────────────────────────────────────────────────

In [ ]:
# ── Part 2: FORCED fallback — bad primary key triggers real failover ──
print("\n\n--- FORCED FALLBACK DEMO ---")
print("Primary target uses a deliberately invalid Groq API key.")
print("Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.\n")

forced_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "provider": "groq",
            "api_key": "gsk_INVALID_KEY_THIS_WILL_FAIL",   #  401 from Groq
            "override_params": {"model": "llama-3.3-70b-versatile"}
        },
        {"override_params": {"model": GROQ_OTHER_MODEL}}         # real key via virtual key
    ]
}

portkey_forced = Portkey(api_key=PORTKEY_API_KEY, config="pc-force-2f52ca")

try:
    t0 = time.time()
    r = portkey_forced.chat.completions.create(
        messages=[{"role": "user", "content": "What is ge ai?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Got a response in {ms:.0f}ms despite the bad primary key!")
    print(f"   {r.choices[0].message.content[:250]}")
    print("\n→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED")
    print("   The fallback fired automatically — app code never saw the error.")
except Exception as e:
    print(f"❌ Both targets failed: {e}")



--- FORCED FALLBACK DEMO ---
Primary target uses a deliberately invalid Groq API key.
Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.

✅ Got a response in 855ms despite the bad primary key!
   GE AI refers to the artificial intelligence (AI) technologies and solutions developed by General Electric (GE), a multinational conglomerate company. GE AI focuses on developing AI and machine learning (ML) capabilities that can be applied across var

→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED
   The fallback fired automatically — app code never saw the error.


## Load Balancing

In [9]:
portkey_lb = Portkey(api_key=PORTKEY_API_KEY, config="pc-force-2f52ca")

section("Load Balancing (70% large / 30% small)")

lb_questions = [
    "Why is the sky blue?",
    "What causes ocean tides?",
    "How does caffeine affect the brain?",
    "Define inflation in one sentence.",
    "What is quantum computing?",
    "What is the Fibonacci sequence?"
]

print("Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).\n")

for i, q in enumerate(lb_questions, 1):
    try:
        t0 = time.time()
        r = portkey_lb.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        print(f"Req {i} [{ms:.0f}ms]: {q}")
        print(f"         {r.choices[0].message.content[:120]}...")
    except Exception as e:
        print(f"Req {i}: ERROR — {e}")

print("\n✅ Check Portkey Logs to see which provider served each request")
print("   Set weight=0 to pause a target without removing it from the config")
print(f"Req {i} [{ms:.0f}ms] ({r.model}): {q}")




  Load Balancing (70% large / 30% small)
Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).

Req 1 [1170ms]: Why is the sky blue?
         The sky appears blue to us because of a phenomenon called scattering, which is the way that light behaves when it encoun...
Req 2 s]: What causes ocean tides?
         Ocean tides are primarily caused by the gravitational pull of the Moon and, to a lesser extent, the Sun on the Earth's o...
Req 3 s]: How does caffeine affect the brain?
         Caffeine is a stimulant that affects the brain in several ways. Here are the key effects:

1. **Blocks Adenosine**: Aden...
Req 4 s]: Define inflation in one sentence.
         Inflation is a sustained increase in the general price level of goods and services in an economy over a period of time, ...
Req 5 [1196ms]: What is quantum computing?
         Quantum computing is a new and rapidly evolving field of computing that uses the principles of quantum mechanics to per

## Caching

In [10]:
# Portkey Dashboard'dan aldığınız Cache Config Slug ID'sini buraya yazın:
portkey_cached = Portkey(api_key=PORTKEY_API_KEY, config="pc-ai-sec-bd2117")

section("Portkey Prompt Caching")

q = "Define Kubernetes ConfigMap in one sentence."

call_params = dict(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
    max_tokens=120,
)

# -------------------------------------------------------------
# 1st Request: CACHE MISS (Not in cache, routes to LLM -> slow)
# -------------------------------------------------------------
t0 = time.time()
r1 = portkey_cached.chat.completions.create(**call_params)
ms1 = (time.time() - t0) * 1000

print(f"1st Request (Cache MISS) : ⏱ {ms1:.0f}ms")
print(f"   A: {r1.choices[0].message.content}\n")


# -------------------------------------------------------------
# 2nd Request: CACHE HIT (Served from cache -> instant/very fast)
# -------------------------------------------------------------
t0 = time.time()
r2 = portkey_cached.chat.completions.create(**call_params)
ms2 = (time.time() - t0) * 1000

speedup = ms1 / max(ms2, 1)
print(f"2nd Request (Cache HIT)  : ⏱ {ms2:.0f}ms  🚀 ({speedup:.1f}x faster!)")
print(f"   A: {r2.choices[0].message.content}")

print("\n Check Portkey Dashboard -> Logs:")
print("   - 1st Request: Cache Miss (LLM invoked, normal latency)")
print("   - 2nd Request: Cache Hit (0 token cost, instant response)")




  Portkey Prompt Caching
1st Request (Cache MISS) : ⏱ 689ms
   A: A Kubernetes ConfigMap is a resource that stores and manages configuration data, such as environment variables, configuration files, and other settings, for applications running in a Kubernetes cluster.

2nd Request (Cache HIT)  : ⏱ 287ms  🚀 (2.4x faster!)
   A: A Kubernetes ConfigMap is a resource that stores and manages configuration data, such as environment variables, configuration files, and other settings, for applications running in a Kubernetes cluster.

 Check Portkey Dashboard -> Logs:
   - 1st Request: Cache Miss (LLM invoked, normal latency)
   - 2nd Request: Cache Hit (0 token cost, instant response)


## LangChain Drop-In Integration

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


portkey_llm = ChatOpenAI(
    api_key=PORTKEY_API_KEY,          # Portkey API key (not Groq key)
    base_url=PORTKEY_GATEWAY_URL,     # Portkey gateway endpoint
    model=GROQ_MODEL,                 # "@flight-policsy/llama-3.3-70b-versatile"
    temperature=0,
    default_headers=createHeaders(    # adds x-portkey-* headers
        api_key=PORTKEY_API_KEY,
        metadata={
            "_user":       "langchain-demo",
            "environment": "notebook",
            "feature":     "langchain-integration"
        }
    )
)


section("LangChain Drop-in")

# Test 1: Direct .invoke() — same as calling any LangChain LLM directly
print("--- Test 1: Direct .invoke() ---")
t0 = time.time()
r = portkey_llm.invoke([
    SystemMessage(content="You are an Enterprise IT Assistant."),
    HumanMessage(content="What is the difference between a Deployment and a TESTING?")
])
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(r.content[:250])



  LangChain Drop-in
--- Test 1: Direct .invoke() ---
✅ 1867ms
As an Enterprise IT Assistant, I'd be happy to explain the difference between a deployment and testing.

**Testing:**
Testing refers to the process of evaluating a software application, system, or component to ensure it meets the required specificati


In [12]:
# LCEL chain — prompt | llm | parser
print("\n--- Test 2: LCEL chain ---")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Enterprise IT expert. Be concise."),
    ("human",  "{question}")
])
chain = prompt | portkey_llm | StrOutputParser()

t0 = time.time()
answer = chain.invoke({"question": "Explain Kubernetes pod affinity rules."})
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(answer[:250])

print("\n→ Drop-in replacement for ChatGroq in any LangChain app")
print("→ Every call is now logged in Portkey with metadata, retries, and fallback")


--- Test 2: LCEL chain ---
✅ 1493ms
Kubernetes pod affinity rules define how pods are scheduled and placed on nodes based on their affinity or anti-affinity with other pods. There are two types:

1. **Pod Affinity**: Pods are scheduled on the same node as pods that match the specified 

→ Drop-in replacement for ChatGroq in any LangChain app
→ Every call is now logged in Portkey with metadata, retries, and fallback


In [14]:
PRODUCTION_CONFIG = {
    "strategy":        {"mode": "fallback"},
    "request_timeout": 30000,              # 30s hard cap
    "retry": {
        "attempts":        2,
        "on_status_codes": [429, 500, 503]
    },
    "cache": {"mode": "simple"},           # free repeated queries
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},        # primary — large 70b
        {"override_params": {"model": GROQ_OTHER_MODEL}}   # fallback — small 8b
    ]
}

production_gateway = Portkey(api_key=PORTKEY_API_KEY, config="pc-ai-sec-bd2117")

section("Full Production Gateway")
print("⚙️  Active:")
print("   Fallback  : large Groq 70b → small Groq 8b on failure")
print("   Retry     : 2 attempts on 429/500/503")
print("   Timeout   : 30 seconds")
print("   Cache     : simple exact match")

test_suite = [
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "support-chat",     "How does Intel SRIOV work?"),
    ("carol", "docs-lookup",      "Explain BGP route reflectors."),
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),  # cache hit!
    ("dave",  "support-chat",     "What is a Kubernetes operator pattern?"),
]

for user, feature, q in test_suite:
    try:
        t0 = time.time()
        r = production_gateway.with_options(
            metadata={
                "_user":       user,
                "feature":     feature,
                "session_id":  str(uuid.uuid4())[:8],
                "environment": "production"
            }
        ).chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        cache_hint = " — CACHE HIT!" if ms < 100 else ""
        print(f"\n👤 {user:8s} | {feature:18s} | {ms:.0f}ms{cache_hint}")
        print(f"   Q: {q}")
        print(f"   A: {r.choices[0].message.content[:150]}...")
    except Exception as e:
        print(f"   ERROR: {e}")

print("\n✅ Full observability, resilience, and cost control on every call.")


  Full Production Gateway
⚙️  Active:
   Fallback  : large Groq 70b → small Groq 8b on failure
   Retry     : 2 attempts on 429/500/503
   Timeout   : 30 seconds
   Cache     : simple exact match

👤 alice    | enterprise-rag     | 1589ms
   Q: What is Kubernetes RBAC?
   A: Kubernetes RBAC (Role-Based Access Control) is a mechanism for controlling access to Kubernetes resources based on user roles. It allows administrator...

👤 bob      | support-chat       | 2414ms
   Q: How does Intel SRIOV work?
   A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that enables a single physical device, such as a network interface card (NIC) or a stora...

👤 carol    | docs-lookup        | 2005ms
   Q: Explain BGP route reflectors.
   A: BGP (Border Gateway Protocol) route reflectors are a key component in large-scale BGP networks, designed to improve scalability, reduce configuration ...

👤 alice    | enterprise-rag     | 1409ms
   Q: What is Kubernetes RBAC?
   A: Kubernetes RBAC (R